# Which Olist bronze tables need deduplication?

Checks **rows vs distinct natural key** for every table in the BigQuery bronze dataset
`olin_bronze_dev_jun`, to see which tables actually contain duplicate rows.

There are **two kinds of "need dedup"**:
1. **Append-guard** — the EL (Meltano `target-bigquery`) loads in *append* mode, so re-running it
   duplicates rows in *every* table.
2. **Inherent dupes** — some Olist tables have duplicate rows on their natural key even in a
   single clean load.

This notebook surfaces both via `dupes = rows - distinct_key`.

## 1. Connect to BigQuery

Auth uses a service-account keyfile via `GOOGLE_APPLICATION_CREDENTIALS` (git-ignored under
`secrets/`). The default path assumes this notebook lives in `notes/`, so the keyfile is one
level up in `../secrets/`. Adjust if you run it from elsewhere.

In [ ]:
import os
from google.cloud import bigquery

# Service-account auth. Keyfile is git-ignored under secrets/.
os.environ.setdefault(
    "GOOGLE_APPLICATION_CREDENTIALS",
    "../secrets/sctp-team2-project2-elt-1853e88c8665.json",
)

PROJECT  = "sctp-team2-project2-elt"
DATASET  = "olin_bronze_dev_jun"
LOCATION = "US"

client = bigquery.Client(project=PROJECT, location=LOCATION)
print("Connected to", PROJECT)

## 2. Natural key per table

Each table is checked against its natural key. A **unique** key should show `dupes = 0` on a
clean load. Two tables are special:

| table | natural key | note |
|---|---|---|
| orders | order_id | unique |
| customers | customer_id | unique |
| products | product_id | unique |
| sellers | seller_id | unique |
| product_category_name_translation | product_category_name | unique |
| order_items | (order_id, order_item_id) | composite |
| order_payments | (order_id, payment_sequential) | composite |
| order_reviews | review_id / (review_id, order_id) | **review_id repeats** in Olist |
| geolocation | (none) | **no key** — many lat/lng per zip |

Raw bronze columns are all STRING (Singer load), so composite keys are built with `concat` directly.

In [ ]:
checks = [
    ("orders",                            "order_id"),
    ("customers",                         "customer_id"),
    ("products",                          "product_id"),
    ("sellers",                           "seller_id"),
    ("product_category_name_translation", "product_category_name"),
    ("order_items",                       "concat(order_id,'|',order_item_id)"),
    ("order_payments",                    "concat(order_id,'|',payment_sequential)"),
    ("order_reviews",                     "review_id"),
    ("order_reviews",                     "concat(review_id,'|',order_id)"),
    ("geolocation",                       "geolocation_zip_code_prefix"),
]

present = {t.table_id for t in client.list_tables(DATASET)}

print(f"{'table':36s}{'key':34s}{'rows':>10}{'distinct':>12}{'dupes':>8}")
print("-" * 100)
for table, key in checks:
    if table not in present:
        print(f"{table:36s}{key[:32]:34s}{'(missing)':>10}")
        continue
    q = f"select count(*) AS r, count(distinct {key}) AS d from `{PROJECT}.{DATASET}.{table}`"
    row = list(client.query(q, location=LOCATION).result())[0]
    print(f"{table:36s}{key[:32]:34s}{row.r:>10}{row.d:>12}{row.r-row.d:>8}")

## 3. How to read the result

- **`dupes = 0`** → the key is unique in the current data. (After an append re-run, `rows`
  becomes a multiple of `distinct` — that's the append-guard case, not inherent dupes.)
- **`dupes > 0` on a single clean load** → genuine duplicates on that key.

**Expected for Olist:**
- `order_reviews` on `review_id` → **dupes > 0** (review_id repeats); even `(review_id, order_id)`
  may show a few.
- `geolocation` on zip prefix → **huge dupes** (many coordinates per zip, by design — no real key).
- everything else → `dupes = 0` on a clean load.

## 4. What this means for staging

| table | dedup approach in `stg_*` |
|---|---|
| order_reviews | dedup on `(review_id, order_id)` |
| geolocation | `SELECT DISTINCT` (no key) |
| all others | dedup on their unique key (append-guard) |

Each `stg_*` model already implements exactly this (via `QUALIFY row_number() ... order by
`_sdc_sequence desc`, or `SELECT DISTINCT` for geolocation).